# Day 3: Preprocessing and Stratified Data Split

This notebook prepares the credit card fraud dataset for modeling.

- Day 3 prepares the dataset for modeling.
- The goal is to create leakage-safe train/validation/test splits.
- No model training is performed in this notebook.

## Day 3 Objectives

- Load validated raw dataset
- Separate features and target
- Create stratified train/validation/test split
- Preserve rare fraud class distribution
- Build preprocessing pipeline
- Fit preprocessing only on training data
- Transform validation and test data without leakage
- Save processed artifacts

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Add the project root to sys.path so this notebook works whether it is
# run from the notebooks/ folder or from the repository root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_loader import load_credit_card_data
from src.data.split_data import (
    separate_features_target,
    create_train_val_test_split,
    get_split_summary,
)
from src.preprocessing.feature_config import (
    TARGET_COLUMN,
    SCALE_FEATURES,
    PASSTHROUGH_FEATURES,
    ALL_FEATURES,
)
from src.preprocessing.preprocessors import (
    fit_transform_train_val_test,
    get_preprocessing_summary,
)

In [ ]:
DATA_PATH = Path("../data/raw/creditcard.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/creditcard.csv")

print(f"Using dataset path: {DATA_PATH.resolve()}")

## 1. Load Validated Dataset

In [ ]:
df = load_credit_card_data(DATA_PATH, validate=True)

print("Dataset shape:", df.shape)
df.head()

## 2. Feature Groups

- `Class` is the target column.
- `Time` and `Amount` will be scaled.
- `V1`-`V28` are already PCA-transformed and will pass through unchanged.

In [ ]:
print("Target column:", TARGET_COLUMN)
print("Scale features:", SCALE_FEATURES)
print("First 5 passthrough features:", PASSTHROUGH_FEATURES[:5])
print("Total feature count:", len(ALL_FEATURES))

## 3. Separate Features and Target

In [ ]:
X, y = separate_features_target(df)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Is Class in X.columns?", TARGET_COLUMN in X.columns)
y.value_counts(normalize=True)

## 4. Stratified Train/Validation/Test Split

- Fraud is rare, so a random split could easily under- or over-represent it.
- Stratification preserves the original class distribution in every split.
- The split ratio used here is 70% train, 15% validation, 15% test.

In [ ]:
splits = create_train_val_test_split(X, y)

X_train = splits["X_train"]
X_val = splits["X_val"]
X_test = splits["X_test"]
y_train = splits["y_train"]
y_val = splits["y_val"]
y_test = splits["y_test"]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

## 5. Split Summary

In [ ]:
split_summary = get_split_summary(splits)
split_summary

In [ ]:
split_distribution = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": split_summary["train_rows"],
            "legitimate_count": split_summary["train_class_distribution"]["legitimate_count"],
            "fraud_count": split_summary["train_class_distribution"]["fraud_count"],
            "fraud_percentage": split_summary["train_class_distribution"]["fraud_percentage"],
        },
        {
            "split": "validation",
            "rows": split_summary["validation_rows"],
            "legitimate_count": split_summary["validation_class_distribution"]["legitimate_count"],
            "fraud_count": split_summary["validation_class_distribution"]["fraud_count"],
            "fraud_percentage": split_summary["validation_class_distribution"]["fraud_percentage"],
        },
        {
            "split": "test",
            "rows": split_summary["test_rows"],
            "legitimate_count": split_summary["test_class_distribution"]["legitimate_count"],
            "fraud_count": split_summary["test_class_distribution"]["fraud_count"],
            "fraud_percentage": split_summary["test_class_distribution"]["fraud_percentage"],
        },
    ]
)
split_distribution

## 6. Leakage-Safe Preprocessing

- Preprocessing must be fitted only on training data.
- Validation and test sets must only be transformed, never fitted on.
- This prevents information from validation/test leaking into training.

In [ ]:
preprocessing_results = fit_transform_train_val_test(X_train, X_val, X_test)

preprocessor = preprocessing_results["preprocessor"]
X_train_processed = preprocessing_results["X_train_processed"]
X_val_processed = preprocessing_results["X_val_processed"]
X_test_processed = preprocessing_results["X_test_processed"]
feature_names = preprocessing_results["feature_names"]

print("X_train_processed shape:", X_train_processed.shape)
print("X_val_processed shape:", X_val_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)
print("Number of feature names:", len(feature_names))

## 7. Preprocessing Summary

In [ ]:
preprocessing_summary = get_preprocessing_summary(
    X_train,
    X_train_processed,
    feature_names,
)

preprocessing_summary

## 8. Data Leakage Checks

In [ ]:
print("Is Class in X_train_processed.columns?", TARGET_COLUMN in X_train_processed.columns)
print("X_train_processed rows match X_train rows:", X_train_processed.shape[0] == X_train.shape[0])
print("X_val_processed rows match X_val rows:", X_val_processed.shape[0] == X_val.shape[0])
print("X_test_processed rows match X_test rows:", X_test_processed.shape[0] == X_test.shape[0])
print("Missing values in X_train_processed:", int(X_train_processed.isna().sum().sum()))
print("Missing values in X_val_processed:", int(X_val_processed.isna().sum().sum()))
print("Missing values in X_test_processed:", int(X_test_processed.isna().sum().sum()))

## 9. Day 3 Key Findings

- Features and target were separated, with `Class` removed from the feature matrix.
- The stratified split preserved the rare fraud class distribution across train, validation, and test.
- `Time` and `Amount` were scaled using a preprocessor fitted only on training data.
- `V1`-`V28` were passed through unchanged, since they are already PCA-transformed.
- The preprocessor was fitted only on training data, then used to transform validation and test data.
- The dataset is now ready for baseline modeling.

## 10. Day 4 Next Steps

- Train a Logistic Regression baseline.
- Train a Random Forest baseline.
- Evaluate using precision, recall, F1-score, PR-AUC, ROC-AUC, and a confusion matrix.
- Continue avoiding accuracy-only evaluation, since the dataset is highly imbalanced.